# Importing Libraries

In [ ]:
!pip install -q langchain-core langchain-groq

In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# LLM

In [ ]:
api_key = userdata.get("GROQ_API_KEY")

In [ ]:
llm = ChatGroq(
    api_key = api_key,
    model_name = "llama-3.3-70b-versatile",
    temperature = 0.1
)

# Prompt Template


In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# Memory

In [ ]:
chat_history = []

def ask(question):
    global chat_history
    chain = prompt | llm
    response = chain.invoke({
        "chat_history": chat_history,
        "question": question
    })
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response.content))
    return response.content

#  Chat

In [ ]:
import gradio as gr

def gradio_ask(question, history):
    answer = ask(question)
    history.append((question, answer))
    return "", history

def reset():
    global chat_history
    chat_history.clear()
    return [], []

with gr.Blocks(title="Chat-with-Your-PDF") as demo:
    gr.Markdown("# Chat-with-Your-PDF")
    gr.Markdown("Ask anything from the PDF!")

    chatbot = gr.Chatbot(height=450, label="Chat")

    with gr.Row():
        msg      = gr.Textbox(placeholder="Chat...", show_label=False, scale=4)
        send_btn = gr.Button("Send", variant="primary", scale=1)

    reset_btn = gr.Button("🗑️ Clear Memory", variant="secondary")

    send_btn.click(fn=gradio_ask, inputs=[msg, chatbot], outputs=[msg, chatbot])
    msg.submit(fn=gradio_ask, inputs=[msg, chatbot], outputs=[msg, chatbot])
    reset_btn.click(fn=reset, outputs=[chatbot, chatbot])

demo.launch(debug=True)